# Pipeline Sample B

1. Sequence QC (FastQC + fastp)
2. Alignment (BWA-MEM)
3. Alignment post-processing (sort, index, MarkDuplicates, BQSR)
4. Variant calling (GATK HaplotypeCaller)
5. Combined VCF for sample A and sample B (GenomicsDBImport + GenotypeGVCFs)
6. Variant filtering (VQSR + vcftools)

Sample A files (reference index, dbSNP, `A.g.vcf`, etc.) produced by the other notebooks are reused.

---
## ENV setup

In [1]:
import os
os.environ["JAVA_HOME"] = "/Users/noel/miniconda3/envs/bio392/lib/jvm"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ":" + os.environ["PATH"]

In [3]:
%cd /Users/noel/Coding/sample_A/
!pwd

/Users/noel/Coding/sample_A
/Users/noel/Coding/sample_A


---
## 1. Sequence QC

Running FastQC on the raw reads for sample B, then trim/filter with fastp.

### FastQC on raw reads

In [4]:
!fastqc --version

FastQC v0.12.1


In [5]:
!mkdir -p ./data/FASTQ_Chr14/output/FastQC_Out

In [6]:
!fastqc -f fastq -t 2 \
    -o ./data/FASTQ_Chr14/output/FastQC_Out \
    ./data/FASTQ_Chr14/B_1.fq.gz \
    ./data/FASTQ_Chr14/B_2.fq.gz

application/gzip
application/gzip
Started analysis of B_1.fq.gz
Approx 5% complete for B_1.fq.gz
Approx 10% complete for B_1.fq.gz
Started analysis of B_2.fq.gz
Approx 15% complete for B_1.fq.gz
Approx 5% complete for B_2.fq.gz
Approx 20% complete for B_1.fq.gz
Approx 10% complete for B_2.fq.gz
Approx 25% complete for B_1.fq.gz
Approx 15% complete for B_2.fq.gz
Approx 30% complete for B_1.fq.gz
Approx 20% complete for B_2.fq.gz
Approx 35% complete for B_1.fq.gz
Approx 25% complete for B_2.fq.gz
Approx 40% complete for B_1.fq.gz
Approx 30% complete for B_2.fq.gz
Approx 45% complete for B_1.fq.gz
Approx 35% complete for B_2.fq.gz
Approx 50% complete for B_1.fq.gz
Approx 40% complete for B_2.fq.gz
Approx 55% complete for B_1.fq.gz
Approx 45% complete for B_2.fq.gz
Approx 60% complete for B_1.fq.gz
Approx 50% complete for B_2.fq.gz
Approx 65% complete for B_1.fq.gz
Approx 55% complete for B_2.fq.gz
Approx 70% complete for B_1.fq.gz
Approx 60% complete for B_2.fq.gz
Approx 75% complete for 

In [7]:
!ls ./data/FASTQ_Chr14/output/FastQC_Out

A_1_fastqc      A_1_fastqc.zip  A_2_fastqc.zip  B_1_fastqc.zip  B_2_fastqc.zip
A_1_fastqc.html A_2_fastqc.html B_1_fastqc.html B_2_fastqc.html


### Trim/filter with fastp

In [8]:
!fastp --version

fastp 1.3.3


In [9]:
!mkdir -p ./output/fastp

In [10]:
!fastp -i ./data/FASTQ_Chr14/B_1.fq.gz \
    -I ./data/FASTQ_Chr14/B_2.fq.gz \
    -o ./output/fastp/clean_B_1.fq.gz \
    -O ./output/fastp/clean_B_2.fq.gz \
    -q 30 \
    -h ./output/fastp/B_fastp_report.html \
    -j ./output/fastp/B_fastp_report.json

Read1 before filtering:
total reads: 397602
total bases: 40157802
Q20 bases: 39898011(99.3531%)
Q30 bases: 38738731(96.4663%)
Q40 bases: 14612471(36.3876%)

Read2 before filtering:
total reads: 397602
total bases: 40157802
Q20 bases: 39623778(98.6702%)
Q30 bases: 38081683(94.8301%)
Q40 bases: 14121679(35.1655%)

Read1 after filtering:
total reads: 390677
total bases: 39449142
Q20 bases: 39248680(99.4918%)
Q30 bases: 38253784(96.9699%)
Q40 bases: 14566090(36.9237%)

Read2 after filtering:
total reads: 390677
total bases: 39449142
Q20 bases: 38987294(98.8293%)
Q30 bases: 37629533(95.3875%)
Q40 bases: 14086299(35.7075%)

Filtering result:
reads passed filter: 781354
reads failed due to low quality: 13270
reads failed due to too many N: 580
reads failed due to too short: 0
reads failed due to adapter dimer: 0
reads with adapter trimmed: 1190
bases trimmed due to adapters: 18554

Duplication rate: 0.000251508%

Insert size peak (evaluated by paired-end reads): 165

JSON report: ./output/fas

---
## 2. Alignment with BWA-MEM

We can reuse the reference index built during the sample A run.

In [11]:
!mkdir -p ./output/bwa

In [12]:
!bwa mem -M -t 2 \
    -R "@RG\tID:B\tPL:ILLUMINA\tLB:lib1\tSM:B" \
    -o ./output/bwa/B.sam \
    ./reference/chr14.fa \
    ./output/fastp/clean_B_1.fq.gz \
    ./output/fastp/clean_B_2.fq.gz

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 198056 sequences (20000006 bp)...
[M::process] read 198068 sequences (20000018 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (3, 94650, 1, 0)
[M::mem_pestat] skip orientation FF as there are not enough pairs
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (159, 186, 223)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (31, 351)
[M::mem_pestat] mean and std.dev: (193.85, 47.45)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 415)
[M::mem_pestat] skip orientation RF as there are not enough pairs
[M::mem_pestat] skip orientation RR as there are not enough pairs
[M::mem_process_seqs] Processed 198056 reads in 9.815 CPU sec, 4.872 real sec
[M::process] read 198066 sequences (20000096 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (3, 94783, 2, 1)
[M::mem_pestat] skip orientation

Alignment stats:

In [13]:
!samtools flagstat -@ 2 ./output/bwa/B.sam

781493 + 0 in total (QC-passed reads + QC-failed reads)
781354 + 0 primary
139 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
781488 + 0 mapped (100.00% : N/A)
781349 + 0 primary mapped (100.00% : N/A)
781354 + 0 paired in sequencing
390677 + 0 read1
390677 + 0 read2
780190 + 0 properly paired (99.85% : N/A)
781344 + 0 with itself and mate mapped
5 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


---
## 3. Alignment post-processing

SAM &rarr; BAM &rarr; sort &rarr; index &rarr; MarkDuplicates &rarr; BQSR.

### Convert SAM to BAM

In [14]:
!samtools view -bS \
    -@ 2 \
    -o ./output/bwa/B.bam \
    ./output/bwa/B.sam

### Sort BAM by coordinate

In [15]:
!samtools sort -@ 2 \
    -o ./output/bwa/B_sort.bam \
    ./output/bwa/B.bam

[bam_sort_core] merging from 0 files and 2 in-memory blocks...


### Index sorted BAM

In [16]:
!samtools index -@ 2 ./output/bwa/B_sort.bam

### Mark duplicates (GATK / Picard)

In [17]:
!gatk --java-options "-Xmx4G" MarkDuplicates \
    -I ./output/bwa/B_sort.bam \
    -O ./output/bwa/B_sort_markdup.bam \
    -M ./output/bwa/B_sort_markdup_metrics.txt \
    --TMP_DIR .

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx4G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar MarkDuplicates -I ./output/bwa/B_sort.bam -O ./output/bwa/B_sort_markdup.bam -M ./output/bwa/B_sort_markdup_metrics.txt --TMP_DIR .
08:28:26.983 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:28:27.017 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/Users/noel/Coding/sample_A/libgkl_compression767315491211158822.dylib: dlopen(/Users/noel/Coding/sample_A/libgkl_compression767315491211158822.dylib,

### Collect WGS metrics (QC)

In [18]:
!gatk --java-options "-Xmx4G" CollectWgsMetrics \
    -MQ 20 -Q 20 \
    -R ./reference/chr14.fa \
    -O ./output/bwa/B_sort_markdup_WGSmetrics.txt \
    -I ./output/bwa/B_sort_markdup.bam

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx4G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar CollectWgsMetrics -MQ 20 -Q 20 -R ./reference/chr14.fa -O ./output/bwa/B_sort_markdup_WGSmetrics.txt -I ./output/bwa/B_sort_markdup.bam
08:28:41.745 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:28:41.767 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/j3/dr8ff52x0m393lcx6yws1rl80000gn/T/noel/libgkl_compression9495082100032617058.dylib: dlopen(/private/var/folders/j3/dr8ff52x

### Base Quality Score Recalibration (BQSR)

Reference files `.fai` and `.dict` already built in run of sample A.

In [19]:
!mkdir -p ./output/bqsr

In [20]:
!gatk --java-options "-Xmx4G" BaseRecalibrator \
    -R ./reference/chr14.fa \
    -I ./output/bwa/B_sort_markdup.bam \
    -O ./output/bqsr/B_sort_markdup_BQSR.report \
    --known-sites ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    --known-sites ./db/resources_broad_hg38_v0_Homo_sapiens_assembly38.known_indels.vcf.gz \
    --known-sites ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx4G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar BaseRecalibrator -R ./reference/chr14.fa -I ./output/bwa/B_sort_markdup.bam -O ./output/bqsr/B_sort_markdup_BQSR.report --known-sites ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz --known-sites ./db/resources_broad_hg38_v0_Homo_sapiens_assembly38.known_indels.vcf.gz --known-sites ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz
08:29:39.054 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:29:39.065 WARN  NativeLibraryLoader 

In [21]:
!gatk --java-options "-Xmx4G" ApplyBQSR \
    -I ./output/bwa/B_sort_markdup.bam \
    -O ./output/bqsr/B_sort_markdup_recal.bam \
    --bqsr-recal-file ./output/bqsr/B_sort_markdup_BQSR.report

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx4G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyBQSR -I ./output/bwa/B_sort_markdup.bam -O ./output/bqsr/B_sort_markdup_recal.bam --bqsr-recal-file ./output/bqsr/B_sort_markdup_BQSR.report
08:30:11.057 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:30:11.068 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/j3/dr8ff52x0m393lcx6yws1rl80000gn/T/libgkl_compression8005514384782005157.dylib: dlopen(/private/var/folders/j3/dr8

---
## 4. Variant calling with HaplotypeCaller (per-sample GVCF)

In [22]:
!mkdir -p ./output/haplotypecaller
!mkdir -p ./tmp

In [23]:
!gatk --java-options "-Xmx8G" HaplotypeCaller \
    -R ./reference/chr14.fa \
    -I ./output/bqsr/B_sort_markdup_recal.bam \
    -O ./output/haplotypecaller/B.g.vcf \
    -ERC GVCF \
    --tmp-dir ./tmp

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar HaplotypeCaller -R ./reference/chr14.fa -I ./output/bqsr/B_sort_markdup_recal.bam -O ./output/haplotypecaller/B.g.vcf -ERC GVCF --tmp-dir ./tmp
08:30:30.712 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:30:30.724 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/Users/noel/Coding/sample_A/tmp/libgkl_compression8049700275066422132.dylib: dlopen(/Users/noel/Coding/sample_A/tmp/libgkl_compression8049

In [25]:
!bcftools view -H ./output/haplotypecaller/B.g.vcf | head -5

chr14	1	.	N	<NON_REF>	.	.	END=18601023	GT:DP:GQ:MIN_DP:PL	0/0:0:0:0:0,0,0
chr14	18601024	.	C	<NON_REF>	.	.	END=18601029	GT:DP:GQ:MIN_DP:PL	0/0:2:6:2:0,6,73
chr14	18601030	.	T	<NON_REF>	.	.	END=18601038	GT:DP:GQ:MIN_DP:PL	0/0:3:9:3:0,9,109
chr14	18601039	.	G	<NON_REF>	.	.	END=18601040	GT:DP:GQ:MIN_DP:PL	0/0:4:12:4:0,12,152
chr14	18601041	.	A	<NON_REF>	.	.	END=18601042	GT:DP:GQ:MIN_DP:PL	0/0:5:15:5:0,15,190
[main_vcfview] Error: cannot write to (null)


---
## 5. Combined VCF for sample A and sample B

Rebuilding of GenomicsDB with both samples.

### Regenerate the sample-name

In [26]:
!ls ./output/haplotypecaller/*.g.vcf

./output/haplotypecaller/A.g.vcf ./output/haplotypecaller/B.g.vcf


In [27]:
!ls ./output/haplotypecaller/*.g.vcf | xargs -n1 basename \
    | awk '{split($0,a,".");print a[1]}' > ./output/haplotypecaller/input_name.txt
!cat ./output/haplotypecaller/input_name.txt

A
B


In [28]:
!realpath ./output/haplotypecaller/*.g.vcf > ./output/haplotypecaller/input_path.txt
!cat ./output/haplotypecaller/input_path.txt

/Users/noel/Coding/sample_A/output/haplotypecaller/A.g.vcf
/Users/noel/Coding/sample_A/output/haplotypecaller/B.g.vcf


In [29]:
!paste ./output/haplotypecaller/input_name.txt \
    ./output/haplotypecaller/input_path.txt > ./output/haplotypecaller/samples_map.txt
!cat ./output/haplotypecaller/samples_map.txt

A	/Users/noel/Coding/sample_A/output/haplotypecaller/A.g.vcf
B	/Users/noel/Coding/sample_A/output/haplotypecaller/B.g.vcf


### Rebuild the GenomicsDB

`GenomicsDBImport` requires the target workspace not to exist when using `--genomicsdb-workspace-path` (*at least I think so...*), so we remove the previous one first.

In [30]:
!rm -rf ./output/genomeDB

In [31]:
!gatk --java-options "-Xms8G -Xmx16G" GenomicsDBImport \
    --genomicsdb-workspace-path ./output/genomeDB \
    --batch-size 50 \
    -L chr14 \
    --sample-name-map ./output/haplotypecaller/samples_map.txt \
    --tmp-dir ./tmp \
    --reader-threads 2

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xms8G -Xmx16G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenomicsDBImport --genomicsdb-workspace-path ./output/genomeDB --batch-size 50 -L chr14 --sample-name-map ./output/haplotypecaller/samples_map.txt --tmp-dir ./tmp --reader-threads 2
08:32:44.512 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:32:44.525 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/Users/noel/Coding/sample_A/tmp/libgkl_compression695307121161231812.dylib: dlopen(/Users/no

In [32]:
!cat ./output/genomeDB/callset.json

{
  "callsets": [{
    "sample_name": "A",
    "row_idx": "0",
    "idx_in_file": "0",
    "stream_name": "A_stream"
  }, {
    "sample_name": "B",
    "row_idx": "1",
    "idx_in_file": "0",
    "stream_name": "B_stream"
  }]
}

### Joint genotyping, combined VCF for A + B

In [33]:
!mkdir -p ./output/genotypegvcf

In [34]:
!gatk --java-options "-Xmx8G" GenotypeGVCFs \
    -R ./reference/chr14.fa \
    -V gendb://${PWD}/output/genomeDB \
    -O ./output/genotypegvcf/combine.vcf \
    --tmp-dir ./tmp

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -R ./reference/chr14.fa -V gendb:///Users/noel/Coding/sample_A/output/genomeDB -O ./output/genotypegvcf/combine.vcf --tmp-dir ./tmp
08:33:18.682 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:33:18.694 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/Users/noel/Coding/sample_A/tmp/libgkl_compression1976493638943910240.dylib: dlopen(/Users/noel/Coding/sample_A/tmp/libgkl_compression19

Per-sample VCF for B:

In [35]:
!gatk --java-options "-Xmx8G" GenotypeGVCFs \
    -R ./reference/chr14.fa \
    -V ./output/haplotypecaller/B.g.vcf \
    -O ./output/genotypegvcf/B.vcf \
    --tmp-dir ./tmp

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -R ./reference/chr14.fa -V ./output/haplotypecaller/B.g.vcf -O ./output/genotypegvcf/B.vcf --tmp-dir ./tmp
08:33:42.924 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:33:42.935 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/Users/noel/Coding/sample_A/tmp/libgkl_compression1040434106200398302.dylib: dlopen(/Users/noel/Coding/sample_A/tmp/libgkl_compression1040434106200398302.dylib, 

Confirm both samples are present in the combined VCF:

In [36]:
!bcftools view -h ./output/genotypegvcf/combine.vcf | tail -3

##bcftools_viewVersion=1.23.1+htslib-1.23.1
##bcftools_viewCommand=view -h ./output/genotypegvcf/combine.vcf; Date=Tue May  5 08:33:53 2026
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	A	B


In [37]:
!bcftools view -H ./output/genotypegvcf/combine.vcf | head -10

chr14	18601385	.	C	T	68.94	.	AC=1;AF=0.25;AN=4;BaseQRankSum=-1.123;DP=172;ExcessHet=0;FS=23.489;MLEAC=1;MLEAF=0.25;MQ=35.53;MQRankSum=-0.143;QD=1.21;ReadPosRankSum=-0.706;SOR=4.167	GT:AD:DP:GQ:PL	0/0:111,0:111:18:0,18,3445	0/1:50,7:57:77:77,0,1515
chr14	18601430	.	A	C	1495.5	.	AC=2;AF=0.5;AN=4;BaseQRankSum=-5.495;DP=202;ExcessHet=1.7609;FS=175.704;MLEAC=2;MLEAF=0.5;MQ=33.4;MQRankSum=0.693;QD=8;ReadPosRankSum=-0.132;SOR=8.222	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376	0/1:49,25:74:99:640,0,1470
chr14	18601553	.	T	A	7511.5	.	AC=2;AF=0.5;AN=4;BaseQRankSum=-5.011;DP=640;ExcessHet=1.7609;FS=29.013;MLEAC=2;MLEAF=0.5;MQ=37.94;MQRankSum=-4.714;QD=12.91;ReadPosRankSum=-2.075;SOR=2.328	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662	0/1:139,122:261:99:3289,0,4044
chr14	18601712	.	G	T	2797.5	.	AC=2;AF=0.5;AN=4;BaseQRankSum=-3.946;DP=216;ExcessHet=1.7609;FS=21.535;MLEAC=2;MLEAF=0.5;MQ=44.78;MQRankSum=-8.252;QD=13.51;ReadPosRankSum=0.06;SOR=4.426	GT:AD:DP:GQ:PL	0/1:59,58:117:99:1622,0,1919	0/1:48,42:

---
## 6. Variant filtering with VQSR

Two-stage VQSR on the combined A+B VCF: build & apply SNP model, then build & apply INDEL model.

In [38]:
!mkdir -p ./output/vqsr

### Build VQSR model for SNPs

In [39]:
!gatk --java-options "-Xmx8G" VariantRecalibrator \
    -R ./reference/chr14.fa \
    -V ./output/genotypegvcf/combine.vcf \
    --resource:hapmap,known=false,training=true,truth=true,prior=15.0 ./db/vqsr/resources_broad_hg38_v0_hapmap_3.3.hg38.vcf.gz \
    --resource:omni,known=false,training=true,truth=false,prior=12.0 ./db/vqsr/resources_broad_hg38_v0_1000G_omni2.5.hg38.vcf.gz \
    --resource:1000G,known=false,training=true,truth=false,prior=10.0 ./db/vqsr/resources_broad_hg38_v0_1000G_phase1.snps.high_confidence.hg38_chr14roi.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    -an QD -an MQ -an MQRankSum -an ReadPosRankSum -an FS -an SOR \
    -mode SNP \
    --max-gaussians 4 \
    -O ./output/vqsr/vqsr_snp.recal \
    --tranches-file ./output/vqsr/vqsr_snp.tranches

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar VariantRecalibrator -R ./reference/chr14.fa -V ./output/genotypegvcf/combine.vcf --resource:hapmap,known=false,training=true,truth=true,prior=15.0 ./db/vqsr/resources_broad_hg38_v0_hapmap_3.3.hg38.vcf.gz --resource:omni,known=false,training=true,truth=false,prior=12.0 ./db/vqsr/resources_broad_hg38_v0_1000G_omni2.5.hg38.vcf.gz --resource:1000G,known=false,training=true,truth=false,prior=10.0 ./db/vqsr/resources_broad_hg38_v0_1000G_phase1.snps.high_confidence.hg38_chr14roi.vcf.gz --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz -an QD -an MQ -an MQRankSu

### Build VQSR model for INDELs

In [40]:
!gatk --java-options "-Xmx8G" VariantRecalibrator \
    -R ./reference/chr14.fa \
    -V ./output/genotypegvcf/combine.vcf \
    --resource:mills,known=false,training=true,truth=true,prior=12.0 ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    -an QD -an MQRankSum -an ReadPosRankSum -an FS -an SOR -an DP \
    -mode INDEL \
    --max-gaussians 1 \
    -O ./output/vqsr/vqsr_indel.recal \
    --tranches-file ./output/vqsr/vqsr_indel.tranches

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar VariantRecalibrator -R ./reference/chr14.fa -V ./output/genotypegvcf/combine.vcf --resource:mills,known=false,training=true,truth=true,prior=12.0 ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz -an QD -an MQRankSum -an ReadPosRankSum -an FS -an SOR -an DP -mode INDEL --max-gaussians 1 -O ./output/vqsr/vqsr_indel.recal --tranches-file ./output/vqsr/vqsr_indel.tranches
08:34:35.806 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3

### Apply VQSR

In [41]:
!gatk --java-options "-Xmx8G" ApplyVQSR \
    -V ./output/genotypegvcf/combine.vcf \
    --recal-file ./output/vqsr/vqsr_snp.recal \
    -mode SNP \
    --tranches-file ./output/vqsr/vqsr_snp.tranches \
    --truth-sensitivity-filter-level 99.5 \
    --create-output-variant-index true \
    -O ./output/vqsr/combine_snp_recalibrated.vcf.gz

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyVQSR -V ./output/genotypegvcf/combine.vcf --recal-file ./output/vqsr/vqsr_snp.recal -mode SNP --tranches-file ./output/vqsr/vqsr_snp.tranches --truth-sensitivity-filter-level 99.5 --create-output-variant-index true -O ./output/vqsr/combine_snp_recalibrated.vcf.gz
08:34:42.729 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:34:42.740 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/

In [42]:
!gatk --java-options "-Xmx8G" ApplyVQSR \
    -V ./output/vqsr/combine_snp_recalibrated.vcf.gz \
    -mode INDEL \
    --recal-file ./output/vqsr/vqsr_indel.recal \
    --tranches-file ./output/vqsr/vqsr_indel.tranches \
    --truth-sensitivity-filter-level 99.0 \
    --create-output-variant-index true \
    -O ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyVQSR -V ./output/vqsr/combine_snp_recalibrated.vcf.gz -mode INDEL --recal-file ./output/vqsr/vqsr_indel.recal --tranches-file ./output/vqsr/vqsr_indel.tranches --truth-sensitivity-filter-level 99.0 --create-output-variant-index true -O ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
08:34:48.340 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:34:48.351 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compress

### Drop variants that did not PASS the VQSR filters (vcftools)

In [43]:
!vcftools --gzvcf ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz \
    --recode \
    --recode-INFO-all \
    --remove-filtered-all \
    --out ./output/vqsr/combine_indel_snp_recalibrated


VCFtools - 0.1.17
(C) Adam Auton and Anthony Marcketta 2009

Parameters as interpreted:
	--gzvcf ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
	--recode-INFO-all
	--out ./output/vqsr/combine_indel_snp_recalibrated
	--recode
	--remove-filtered-all

Using zlib version: 1.3.1
After filtering, kept 2 out of 2 Individuals
Outputting VCF file...
After filtering, kept 2849 out of a possible 2926 Sites
Run Time = 0.00 seconds


### Compress and index the final filtered VCF

In [44]:
!bgzip -f -c ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf \
    > ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz

In [45]:
!tabix -f -p vcf ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz

### Inspect the final A+B filtered VCF

In [46]:
!bcftools view -h ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz | tail -3

##bcftools_viewVersion=1.23.1+htslib-1.23.1
##bcftools_viewCommand=view -h ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz; Date=Tue May  5 08:35:09 2026
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	A	B


In [47]:
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz | head -10

chr14	18999581	.	C	G	1799.5	PASS	AC=2;AF=0.5;AN=4;BaseQRankSum=-2.938;DP=199;ExcessHet=1.7609;FS=0;MLEAC=2;MLEAF=0.5;MQ=39.01;MQRankSum=1.48;QD=9.18;ReadPosRankSum=-0.466;SOR=0.138;VQSLOD=-8.57;culprit=MQ	GT:AD:DP:GQ:PGT:PID:PL:PS	0|1:72,36:108:99:0|1:18999581_C_G:1288,0,2896:18999581	0|1:70,18:88:99:0|1:18999581_C_G:521,0,2865:18999581
chr14	18999587	.	T	C	1765.5	PASS	AC=2;AF=0.5;AN=4;BaseQRankSum=-6.375;DP=199;ExcessHet=1.7609;FS=0;MLEAC=2;MLEAF=0.5;MQ=38.72;MQRankSum=1.59;QD=8.87;ReadPosRankSum=-0.007;SOR=0.127;VQSLOD=-9.744;culprit=MQ	GT:AD:DP:GQ:PGT:PID:PL:PS	0|1:74,36:110:99:0|1:18999581_C_G:1282,0,2986:18999581	0|1:72,17:89:99:0|1:18999581_C_G:493,0,2951:18999581
chr14	19180623	.	AT	A	545.99	PASS	AC=4;AF=1;AN=4;DP=15;ExcessHet=0;FS=0;MLEAC=4;MLEAF=1;MQ=39.43;QD=25.36;SOR=5.549;VQSLOD=3.48;culprit=QD	GT:AD:DP:GQ:PL	1/1:0,8:8:24:300,24,0	1/1:0,7:7:21:262,21,0
chr14	19180816	.	T	TG	412.74	PASS	AC=4;AF=1;AN=4;DP=11;ExcessHet=0;FS=0;MLEAC=4;MLEAF=1;MQ=33.82;QD=28.73;SOR=2.494;VQSLOD=

In [48]:
!echo 'Variants before filtering:'
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz | wc -l
!echo 'Variants after filtering:'
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz | wc -l

Variants before filtering:
    2926
Variants after filtering:
    2849


---
**Finished**